<a href="https://colab.research.google.com/github/QuintonPang/job-match-finder-rag/blob/collab/explain_results_collab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import torch
from pathlib import Path
from transformers import pipeline

INPUT_PATH = Path("/content/reranked_results.json")
OUTPUT_PATH = Path("/content/explained_results.json")

EXPLANATION_MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

# Tags that describe the WORK ARRANGEMENT or SENIORITY, not an actual
# skill -- a resume can never "have" or "lack" these, so we exclude
# them before checking for gaps at all.
NON_SKILL_TAGS = {
    "senior", "junior", "mid", "lead", "exec", "executive",
    "digital nomad", "full time", "part time", "remote", "freelance",
    "non tech", "technical",
}



def find_skill_gaps(resume_text: str, tags: list[str]) -> list[str]:
    """
    Deterministic, simple check: for each REAL skill tag (after
    excluding generic descriptors), does that word appear anywhere in
    the resume text?

    Why plain string matching instead of asking the LLM:
        Checking "does this exact word appear in this text" is a
        precise lookup task, not a task requiring interpretation or
        judgment -- exactly the kind of thing small LLMs are
        unreliable at (we observed this firsthand: the model missed
        an obvious Golang gap during testing) and plain code does
        perfectly, every time. We reserve the LLM for genuinely fuzzy
        judgment calls, not exact-match lookups.
    """
    resume_lower = resume_text.lower()
    gaps = []

    for tag in tags:
        tag_lower = tag.lower().strip()
        if tag_lower in NON_SKILL_TAGS:
            continue
        if tag_lower not in resume_lower:
            gaps.append(tag)

    return gaps


def build_fit_prompt(resume_text, job_payload):
    return f"""Candidate's resume:
{resume_text[:1500]}

Job posting:
Title: {job_payload.get('position')}
Company: {job_payload.get('company')}
Summary: {job_payload.get('role_summary')}
Tags: {', '.join(job_payload.get('tags', []))}

In 2-3 sentences, explain why this candidate's background fits this specific job. Only use facts from the resume and job posting above. Be specific, not generic.

Explanation:"""


def build_gaps_prompt(resume_text, job_payload):
    return f"""Candidate's resume:
{resume_text[:1500]}

Job posting:
Title: {job_payload.get('position')}
Tags: {', '.join(job_payload.get('tags', []))}

Looking only at the job's tags above, list which ones do NOT clearly appear in the candidate's resume. If all tags are covered, say "No major gaps found." Keep it to one short sentence.

Missing skills:"""


def generate_explanation(generator, resume_text: str, job_payload: dict) -> dict:
    """
    Generates the fit explanation via LLM (genuinely needs judgment/
    interpretation), and finds skill gaps via simple string matching
    (a precise lookup task, better suited to plain code than an LLM
    call -- see find_skill_gaps() docstring for why).
    """
    fit_prompt = build_fit_prompt(resume_text, job_payload)
    fit_result = generator(fit_prompt, max_new_tokens=120, do_sample=False, repetition_penalty=1.1)
    fit_text = fit_result[0]["generated_text"][len(fit_prompt):].strip().split("\n")[0]

    tags = job_payload.get("tags", [])
    gaps = find_skill_gaps(resume_text, tags)
    gaps_text = ", ".join(gaps) if gaps else "No major gaps found."

    return {
        "fit_explanation": fit_text,
        "skill_gaps": gaps_text,
    }


print(f"Loading reranked results from: {INPUT_PATH}")
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

resume_text = data["resume_text"]
results = data["results"]
print(f"Loaded {len(results)} reranked job results.")

print(f"Loading explanation model '{EXPLANATION_MODEL_NAME}' on GPU...")
generator = pipeline(
    "text-generation",
    model=EXPLANATION_MODEL_NAME,
    device_map="auto",
    dtype=torch.float16,
)
print("Model loaded successfully.")


Loading reranked results from: /content/reranked_results.json
Loaded 5 reranked job results.
Loading explanation model 'microsoft/Phi-3-mini-4k-instruct' on GPU...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Model loaded successfully.


In [2]:
explained = []
for i, item in enumerate(results[:1], start=1):
    payload = item["payload"]
    print(f"Generating explanation {i}/{len(results)}: {payload.get('position')} @ {payload.get('company')}")
    explanation = generate_explanation(generator, resume_text, payload)
    explained.append({
        "payload": payload,
        "rerank_score": item["rerank_score"],
        **explanation,
    })

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(explained, f, indent=2)

print(f"\nSaved explained results to: {OUTPUT_PATH}")
for rank, item in enumerate(explained, start=1):
    p = item["payload"]
    print(f"\n#{rank}  {p.get('position')} @ {p.get('company')}")
    print(f"    Why it fits: {item['fit_explanation']}")
    print(f"    Skill gaps:  {item['skill_gaps']}")

[transformers] Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating explanation 1/5: Senior Backend Engineer Studio AI @ Creative Fabrica


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



Saved explained results to: /content/explained_results.json

#1  Senior Backend Engineer Studio AI @ Creative Fabrica
    Why it fits: The experienced Python developer has a strong foundation in developing backends which aligns well with the role requiring development of scalable server-side apps via Node.js as mentioned in the job description. Their experience can be leveraged to manage subscription services and handle large volumes of artist/craftsperson related data efficiently on an international scale.
    Skill gaps:  golang, engineer
